# Quickstart: Multimodal OpenAI Prompter with Responses API

Use OpenAI's Responses API to enrich your data with multimodal content such as PDFs, images, and documents. This quickstart shows how to use `OpenAIPrompt` with Spark DataFrame columns that contain file paths, HTTP(S) URLs, or base64-encoded data URLs.

### Office documents

Word (`.doc`, `.docx`), Excel (`.xls`, `.xlsx`), and PowerPoint (`.ppt`, `.pptx`) are supported with `apiType="responses"`, subject to your endpoint and model's capabilities. Use a file path or a data URL with the correct Office MIME type. Chat Completions still rejects binary Office attachments.

For file paths, existing content detection runs first, and the Office extension determines routing only when no other type is detected. HTTP(S) URLs follow these path rules: the URL path needs a supported Office extension, and response `Content-Type` headers are not used. HTTP(S) URLs containing query strings are not supported: the Hadoop-based reader encodes `?` into the request path rather than sending query parameters. Remove optional query parameters. For signed or extensionless links, download the file to a correctly named, Spark-accessible path that executors can read (for example, an ADLS `abfss://` path), or use a correctly typed data URL instead. Content detected as HTML is sent as text, even with an Office extension; not all HTML variants are recognized, and other detected markup remains unsupported. Data URLs instead use their declared MIME type without content detection, so the label must match the bytes.

SynapseML does not validate the Office document structure; mislabeled, corrupted, or encrypted documents may be uploaded and rejected by the provider. Office bytes are always forwarded unchanged, including hidden content and macros; SynapseML neither executes nor strips macros, even in a macro-enabled package named or labeled as a supported format. Format checks apply to extensions and MIME labels, not to package contents, and are not a macro filter. Macro-enabled OOXML extensions (`.docm`, `.xlsm`, `.pptm`), template extensions, and generic ZIP/OLE MIME types are not supported; renaming a file does not convert it to a valid Office document. RTF saved as `.doc` is not binary Word and remains unsupported; the existing text sniffer may report it as `application/json`. Convert it to a genuine `.docx` or PDF instead.

The [provider's direct-file input guide](https://developers.openai.com/api/docs/guides/file-inputs) describes text-only processing for non-PDF documents, without embedded images or charts, and up to the first 1,000 rows per spreadsheet sheet. Direct file inputs must be under 50 MB each and within the 50 MB combined request limit; the service enforces both limits. SynapseML's optional per-file check applies only when `fileSizeLimitMB` is set and uses MiB (1,048,576 bytes). Choose a conservative value below the provider limit, such as `10.0` in the example below. This setting does not change the provider's limits. Larger Files API upload limits do not increase these direct-input limits. Convert to PDF when visual document content matters.

## Prerequisites
- An active Spark session (`spark`)
- Access to an OpenAI resource with the Responses API enabled. Deployment should be a multimodal model.
- At least one image, PDF, text, or supported Office file that you want the model to analyze.

## Fill in service information

Next, edit the cell in the notebook to point to your service. In particular set the `service_name`, `deployment_name`, `location`, and `key` variables to match them to your OpenAI service:

In [ ]:
from synapse.ml.core.platform import find_secret

# Fill in the following lines with your service information
# Learn more about selecting which embedding model to choose: https://openai.com/blog/new-and-improved-embedding-model
service_name = "synapseml-openai-3"
deployment_name = "gpt-6-luna"
api_version = (
    "2025-04-01-preview"  # Responses API is only supported in this version and later
)

key = find_secret(
    secret_name="openai-api-key-3", keyvault="mmlspark-build-keys"
)  # please replace this line with your key as a string

assert key is not None and service_name is not None

## Build a multimodal dataset
Each row combines a natural-language question with one attachment. The examples below use HTTP(S) image, PDF, and text URLs plus an inline base64 data URL. To read from Lakehouse, ADLS, or another Spark-accessible file system, replace an attachment value with a Hadoop-compatible path such as `abfss://container@account.dfs.core.windows.net/Files/report.pdf`.

In [ ]:
inline_image_url = (
    "data:image/png;base64,"
    "iVBORw0KGgoAAAANSUhEUgAAAAEAAAABCAQAAAC1HAwCAAAAC0lEQVR42mP8/x8AAusB9Y9Zl1sAAAAASUVORK5CYII="
)

qa_df = spark.createDataFrame(
    [
        (
            "What's in the image?",
            "https://mmlspark.blob.core.windows.net/datasets/OCR/test2.png",
        ),
        (
            "Summarize this document.",
            "https://mmlspark.blob.core.windows.net/datasets/OCR/paper.pdf",
        ),
        (
            "What are the related works to SynapseML?",
            "https://mmlspark.blob.core.windows.net/publicwasb/Overview.md",
        ),
        (
            "Describe this inline image.",
            inline_image_url,
        ),
    ]
).toDF("questions", "attachments")

## Configure `OpenAIPrompt` for file-aware responses
Any column declared as `path` is treated as an attachment. `OpenAIPrompt` reads the value, detects its content type, and adds the appropriate Responses API content part. Set `fileSizeLimitMB` to reject oversized attachments with a row-level error instead of failing the Spark job.

> **Spark AI Functions implementation note:** SynapseML's internal Spark AI Functions use this `OpenAIPrompt` Responses path. For image path, URL, or data-URL columns, `OpenAIPrompt` sends an `input_image` part with a base64 data URL. It does not pass nested low-level message structs to `OpenAIResponses`. Direct `OpenAIResponses` users can instead provide `Array[Struct]` content parts such as `input_text`, `input_image`, and `input_file`.

In [ ]:
from synapse.ml.services.openai import OpenAIPrompt

prompt_template = "{questions}: {attachments}"

prompter = (
    OpenAIPrompt()
    .setSubscriptionKey(key)
    .setDeploymentName(deployment_name)
    .setCustomServiceName(service_name)
    .setApiVersion(api_version)
    .setApiType(
        "responses"
    )  # We need to use the Responses API, instead of Chat Completions API.
    .setPromptTemplate(prompt_template)
    .setColumnTypes(
        {"attachments": "path"}
    )  # Treat every value in the attachment column as a file source
    .setFileSizeLimitMB(10.0)
    .setErrorCol("error")
    .setOutputCol("outputCol")
)

## Generate multimodal responses
Run the transformer and inspect the question, attachment reference, generated response, and row-level error together. A successful row has a populated `outputCol` and a null `error`.

In [ ]:
responses_df = prompter.transform(qa_df)

responses_df.select(
    "questions",
    "attachments",
    "outputCol",
    "error",
).show(truncate=False)